# Signal Extraction Pipeline 


###   Signal Extraction Core (Mandatory)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import caiman as cm
from caiman.source_extraction.cnmf import cnmf, params
from caiman.utils.visualization import plot_contours, nb_view_patches, nb_plot_contour
from caiman.utils.visualization import nb_view_quilt
import logging
from caiman_setup import generate_caiman_panel

# Logger
logger = logging.getLogger('caiman')
logger.setLevel(logging.WARNING)
# 
# Environment variables 
import os
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"


def update_globals(loaded_movie, chosen_input, chosen_output):
    global movie, movie_path, output_path
    movie = loaded_movie
    movie_path = chosen_input
    output_path = chosen_output


# 3. Call script to choose panel
panel = generate_caiman_panel(update_globals)
panel


In [ ]:
# CREATE MEMMAP directly
memmap_file = cm.save_memmap([movie_path], base_name='memmap_', order='C', border_to_0=0)

print(f"Memmap created: {memmap_file}")

# Load in the correct format for CNMF
Yr, dims, num_frames = cm.load_memmap(memmap_file)
images = np.reshape(Yr.T, [num_frames] + list(dims), order='F')
print(f"Shape: {images.shape}")

In [ ]:
# Dataset parameters (ADJUST ACCORDING TO YOUR DATA!)

fr = 1.031              # frame rate in Hz


# Quality parameters (for evaluation) >> This could be altered depending on the segmentation
min_SNR = 3           # validation metric
rval_thr = 0.95         # 0.85


#######################################################################################

# CNMF spatial parameters (VERY IMPORTANT - adjust to your video FOV if it is bigger than 512 pixels)
rf = 15                 # half-width of patches in pixels (patch = 2*rf+1) 
stride_cnmf = 10        # overlap between patches
K = 4                   # expected number of components per patch
gSig = np.array([7, 7]) # expected half-width of GBM in pixels approximation
gSiz = 2 * gSig + 1     # Gaussian kernel size


# Parameter dictionary 
parameter_dict = {
    'data': {
        'fnames': movie_path,
        'fr': fr,
    },
    'init': {
        'K': K,
        'gSig': gSig,
        'gSiz': gSiz,
        'method_init': 'greedy_roi',
        'rolling_sum': True,
        'ssub': 1,
        'tsub': 1,
        'nb': 1,               # number of global background components
    },
    'patch': {
        'rf': rf,
        'stride': stride_cnmf,
        'only_init': False,
    },
    'preprocess': {
        'p': 0,                # deconvolution is not used, postprocessing is made using Matlab Peak detection
    },
    'temporal': {
        'bas_nonneg': True,
    },
    'merging': {
        'merge_thr': 0.85,
    },
    'quality': {
        'min_SNR': min_SNR,    
        'rval_thr': rval_thr,
        'use_cnn': False,
    }
}



# Create parameters object
parameters = params.CNMFParams(params_dict=parameter_dict)

In [ ]:
#Activate parallel computing

import psutil
from caiman.cluster import setup_cluster

num_processors_to_use = None  # None = use all but 1

# Setup cluster
_, cluster, n_processes = setup_cluster(backend='multiprocessing',n_processes=num_processors_to_use)
print(f"Cluster iniciado com {n_processes} processos")

In [ ]:
# Initialize CNMF model
cnmf_model = cnmf.CNMF(n_processes, params=parameters, dview=cluster)

# Run CNMF
cnmf_model.fit(images)

In [ ]:
# Re-run CNMF (Refinement)
cnmf_refit = cnmf_model.refit(images, dview=cluster)


# Evaluate components (SNR, Spatial Correlation, witHout CNN) >> 2 FILTERS (DOUBLE CHECKING BEFORE MANUAL CURATION )
cnmf_refit.estimates.evaluate_components(images, cnmf_refit.params, dview=cluster)

print(f"Accepted Components: {len(cnmf_refit.estimates.idx_components)}")
print(f"Rejected Components: {len(cnmf_refit.estimates.idx_components_bad)}")

In [ ]:
#Correlation image (vizualize)
correlation_image = cm.local_correlations(movie, swap_dim=False)
correlation_image[np.isnan(correlation_image)] = 0

plt.figure(figsize=(8, 6))
plt.imshow(correlation_image, cmap='viridis')
plt.colorbar(label='Correlation')
plt.title(' Correlation Image')
plt.xlabel('X (pixels)')
plt.ylabel('Y (pixels)')
plt.tight_layout()
plt.show()

In [ ]:
# View outlines MAP of accepted vs. rejected components

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection
import caiman as cm
from caiman.source_extraction.cnmf import cnmf, params

fig, ax = plt.subplots(figsize=(12, 10))

ax.imshow(correlation_image, cmap='gray', alpha=0.5)

# Extract A
A = cnmf_refit.estimates.A
if hasattr(A, 'toarray'):
    A = A.toarray()

A_2d = A.reshape((dims[0], dims[1], -1), order='F')

def get_contour_from_spatial(spatial):
    """
    Extracts contour points using ConvexHull from ANY pixel that has a non-zero 
    value in the spatial matrix (No arbitrary thresholds).
    """
    # Coordinates for all functional units found (> 0)
    y_indices, x_indices = np.where(spatial > 0)
    
    if len(y_indices) < 3:  # At least 3 points to draw with ConvexHull
        return None
    
    try:
        from scipy.spatial import ConvexHull
        points = np.column_stack([x_indices, y_indices])
        hull = ConvexHull(points)
        hull_points = points[hull.vertices]
        return hull_points
    except:
        # Fallback se o ConvexHull falhar
        return np.column_stack([x_indices, y_indices])

# ACCEPTED - green
for idx, i in enumerate(cnmf_refit.estimates.idx_components):
    spatial = A_2d[:, :, i]
    contour = get_contour_from_spatial(spatial)
    
    if contour is not None:
        polygon = Polygon(contour, closed=True, 
                          facecolor='none', 
                          edgecolor='lime', 
                          linewidth=2, 
                          alpha=0.8)
        ax.add_patch(polygon)
        
        cx, cy = contour[:, 0].mean(), contour[:, 1].mean()
        ax.text(cx, cy, str(i), color='white', fontsize=8, 
                ha='center', va='center', fontweight='bold')

# REJECTED - red 
for i in cnmf_refit.estimates.idx_components_bad:
    spatial = A_2d[:, :, i]
    contour = get_contour_from_spatial(spatial)
    
    if contour is not None:
        polygon = Polygon(contour, closed=True,
                          facecolor='none',
                          edgecolor='red',
                          linewidth=1,
                          linestyle='--',
                          alpha=0.5)
        ax.add_patch(polygon)

ax.set_xlim(0, dims[1])
ax.set_ylim(dims[0], 0)  
ax.set_title(f'CNMF Spatial Footprints (ConvexHull of Total Component Area)\nGreen=Accepted ({len(cnmf_refit.estimates.idx_components)}), Red=Rejected ({len(cnmf_refit.estimates.idx_components_bad)})')
ax.set_xlabel('X (pixels)')
ax.set_ylabel('Y (pixels)')
plt.tight_layout()
plt.show()

In [ ]:
#Individual SNRs and spatial correlation of each cell

import holoviews as hv
hv.extension('bokeh')
cnmf_refit.estimates.hv_view_components(img=correlation_image, idx=cnmf_refit.estimates.idx_components)

In [ ]:
# Manual Curation of data (you can choose which traces you want to us.The ID is according with above numeration)

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# Original Traces and initial list of accepted traces
C = cnmf_refit.estimates.C
idx_accepted_init = list(cnmf_refit.estimates.idx_components)
num_filtered = len(idx_accepted_init)
PAGE_SIZE = 5

# State dictionary: maps only the components that were already accepted
selection_state = {i: True for i in idx_accepted_init}

# Create interface components
output_chart = widgets.Output()
output_widgets = widgets.Output()

# Slider to navigate only through the pages of already filtered components
slider_page = widgets.IntSlider(
    value=0, 
    min=0, 
    max=max(0, (num_filtered - 1) // PAGE_SIZE), 
    description='Page:',
    continuous_update=False
)

def update_screen(*args):
    page = slider_page.value
    start = page * PAGE_SIZE
    end = min(start + PAGE_SIZE, num_filtered)
    
    # Get the subset of real IDs for this page
    page_indices = idx_accepted_init[start:end]
    num_plots = len(page_indices)
    
    with output_chart:
        clear_output(wait=True)
        if not page_indices:
            print("No components on this page.")
            return
            
        # View traces (stacks) 
        fig, axes = plt.subplots(num_plots, 1, figsize=(11, 2 * num_plots), sharex=True)
        
        if num_plots == 1:
            axes = [axes]
            
        for idx, (i, ax) in enumerate(zip(page_indices, axes)):
            # Blue if kept, Gray if to be removed
            color = "dodgerblue" if selection_state[i] else "lightgray"
            alpha = 1.0 if selection_state[i] else 0.4
            
            ax.plot(C[i], color=color, alpha=alpha, linewidth=1.2)
            
            # ID of each subplot
            ax.text(0.01, 0.85, f"ID {i}", transform=ax.transAxes, fontsize=10, fontweight="bold")
            
            ax.tick_params(axis='y', labelsize=8)
            
            ymin, ymax = np.min(C[i]), np.max(C[i])
            yrange = (ymax - ymin) if (ymax - ymin) > 0 else 1
            ax.set_ylim(ymin - yrange * 0.1, ymax + yrange * 0.1)
            
        axes[0].set_title(f"Review: Filtered Components {start} to {end-1} of {num_filtered} (Blue = Keep)", fontsize=12)
        
        plt.tight_layout()
        plt.show()

    # Generate Checkboxes 
    with output_widgets:
        clear_output(wait=True)
        checkboxes = []
        
        for i in page_indices:
            chk = widgets.Checkbox(value=selection_state[i], description=f"Keep ID {i}", indent=False)
            
            def create_handler(index):
                return lambda change: [selection_state.update({index: change['new']}), update_screen()]
            
            chk.observe(create_handler(i), names='value')
            checkboxes.append(chk)
            
        # Organize into rows of 4 columns
        button_rows = [widgets.HBox(checkboxes[j:j+4]) for j in range(0, len(checkboxes), 4)]
        display(widgets.VBox(button_rows))

slider_page.observe(update_screen, names='value')
display(slider_page, output_chart, output_widgets)
update_screen()

In [ ]:
cm.stop_server(dview=cluster)
_, cluster, n_processes = cm.cluster.setup_cluster(backend='multiprocessing', 
                                                   n_processes=num_processors_to_use, 
                                                   single_thread=False)

### QC Movie Reconstruction 

In [ ]:
import os
import caiman as cm
import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec
from PIL import Image

C = cnmf_refit.estimates.C  # denoised calcium traces
A = cnmf_model.estimates.A
A = cnmf_refit.estimates.A.toarray()
f = cnmf_model.estimates.f
b = cnmf_model.estimates.b

num_pixels, num_gbm = A.shape
num_frames = C.shape[1]
dims = (movie.shape[1], movie.shape[2])  # height, width

# Reconstruction
AC = A @ C  # signal component
bf = b @ f  # background
reconstructed = AC + bf

#########################################################

# Reshape to (frames, height, width)
AC_3D = AC.reshape(dims + (num_frames,), order="F").transpose(2, 0, 1)
bf_3D = bf.reshape(dims + (num_frames,), order="F").transpose(2, 0, 1)
recon_3D = reconstructed.reshape(dims + (num_frames,), order="F").transpose(
    2, 0, 1
)

# Load original
movie_orig = np.array(cm.load(movie_path))
residual_3D = movie_orig - recon_3D

print(f"\nOriginal: {movie_orig.shape}")
print(f"GBM Signal (AC): {AC_3D.shape}")
print(f"Background: {bf_3D.shape}")
print(f"Reconstructed: {recon_3D.shape}")
print(f"Residual: {residual_3D.shape}")

# Subfolder with reconstructed frames
frames_dir = os.path.join(output_path, "output_video")
os.makedirs(frames_dir, exist_ok=True)

# Limit frames to avoid long runtimes (adjust as needed)
max_frames = min(5000, num_frames)

# Normalization
vmin_orig = np.percentile(movie_orig[:max_frames], 1)
vmax_orig = np.percentile(movie_orig[:max_frames], 99.9)
vmin_gbm = np.percentile(AC_3D[:max_frames], 1)
vmax_gbm = np.percentile(AC_3D[:max_frames], 99.9)

################################################################

print(f"\nCreating {max_frames} frames in: {frames_dir}")

for t in range(max_frames):
    fig = plt.figure(figsize=(16, 4))
    gs = GridSpec(1, 4, figure=fig, wspace=0.05)

    # Original
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.imshow(movie_orig[t], cmap="gray", vmin=vmin_orig, vmax=vmax_orig)
    ax1.set_title("Original Movie (Y)", fontsize=10, fontweight="bold")
    ax1.axis("off")

    # Signal component
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.imshow(AC_3D[t], cmap="gray", vmin=vmin_gbm, vmax=vmax_gbm)
    ax2.set_title("Glioblastoma Cells (AC)", fontsize=10, fontweight="bold")
    ax2.axis("off")

    # Reconstructed
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.imshow(recon_3D[t], cmap="gray", vmin=vmin_orig, vmax=vmax_orig)
    ax3.set_title(
        "Reconstructed Movie: AC + bf", fontsize=10, fontweight="bold"
    )
    ax3.axis("off")

    # Residual
    ax4 = fig.add_subplot(gs[0, 3])
    ax4.imshow(residual_3D[t], cmap="gray", vmin=vmin_orig, vmax=vmax_orig)
    ax4.set_title("Residuals (Y-AC-bf)", fontsize=10, fontweight="bold")
    ax4.axis("off")

    fig.suptitle(f"Frame {t}/{num_frames}", fontsize=12, y=0.98)

    # Save PNG
    frame_path = os.path.join(frames_dir, f"frame_{t:04d}.png")
    plt.savefig(frame_path, dpi=100, bbox_inches="tight", pad_inches=0.1)
    plt.close(fig)

    if t % 50 == 0:
        print(f"  Frame {t}/{max_frames}")

In [ ]:
#  COMPILE AVI VIDEO

# COMPILE AVI VIDEO
print("\nCompiling video...")

# Find frame_0 at the correct frames path
first_frame_path = os.path.join(frames_dir, "frame_0000.png")
frame_0 = cv2.imread(first_frame_path)
height, width, _ = frame_0.shape

fourcc = cv2.VideoWriter_fourcc(*"XVID")

#  video output path at the root of the outputs folder
video_output_path = os.path.join(output_path, "cnmf_reconstruction.avi")
out = cv2.VideoWriter(video_output_path, fourcc, 20, (width, height))

for t in range(max_frames):
    # Read each frame from the correct location
    current_frame_path = os.path.join(frames_dir, f"frame_{t:04d}.png")
    frame = cv2.imread(current_frame_path)
    out.write(frame)
    if t % 50 == 0:
        print(f"  Frame {t}/{max_frames}")

out.release()
print(f"Video saved successfully at:\n   {video_output_path}")

### Save files for Connectivity Analysis (to be used in MATLAB)


#### 1. Save scripts for Connectivity Analysis based on peak (Traces, using Masks from CaIman & Traces with from ROI pixel average )

##### 1.1. Masks 

In [ ]:
import numpy as np
from scipy import io
import matplotlib.pyplot as plt
import os

# Masks to Extract ROI

idx_accepted_final = [id_trace for id_trace, keep in selection_state.items() if keep]
cnmf_refit.estimates.idx_components = np.array(idx_accepted_final) # updated list after data curation

A_accepted = cnmf_refit.estimates.A[:, cnmf_refit.estimates.idx_components]


dims = cnmf_refit.estimates.dims
num_cells = A_accepted.shape[1]

print(f"Converting matrix A into {num_cells} fixed masks...")

max_projec = movie.max(axis=0)

# Mask List
mask_list = []

for i in range(num_cells):
    # FIX: Added order='F' to prevent the mask from being rotated/flipped
    component_2d = A_accepted[:, i].toarray().reshape(dims, order='F')
    
    mask_bin = component_2d > 1e-4
    mask_list.append(mask_bin)

###### Visualization of CaIman Masks (Optional)

In [ ]:
# Masks Visualization (optional)
fig, ax = plt.subplots(1, 2, figsize=(14, 7))

ax[0].imshow(max_projec, cmap="gray")
ax[0].set_title("Original Video (Maximum Projection)")
ax[0].axis("off")

ax[1].imshow(max_projec, cmap="gray")
ax[1].set_title(f"CaImAn Masks Overlaid ({num_cells} ROIs)")
ax[1].axis("off")

for idx, mask_bin in enumerate(mask_list):
    ax[1].contour(mask_bin, levels=[0.5], colors="cyan", linewidths=1.0)

plt.tight_layout()
plt.show()

##### 1.2 Mean Intensities

In [ ]:
# EXTRACT Mean Using Masks of CaImAn

num_frames = movie.shape[0]
caiman_pure_means = []

# Run binary masks (segmented with CaIman) of mask_list
for idx, mask_bin in enumerate(mask_list):
    roi_temporal_means = []
    # Run original movie frame by frame
    for t in range(num_frames):
        frame_atual = movie[t]
        
        pixels_no_roi = frame_atual[mask_bin]
        
        if pixels_no_roi.size > 0:
            # aritmatic mean 
            media_pura = int(np.round(pixels_no_roi.mean()))
            roi_temporal_means.append(media_pura)
        else:
            roi_temporal_means.append(0.0)
            
    caiman_pure_means.append(roi_temporal_means)

caiman_pure_means = np.array(caiman_pure_means, dtype=int)
print(f"Conclude Extraction. Matrix with format: {caiman_pure_means.shape}")

##### 1.3 Save Scripts ( benckmark format: (_center.txt); (_mean intensities.txt); (_max projection.png) )

In [ ]:
import os
import imageio.v3 as iio
import numpy as np
from skimage.measure import label, regionprops


def save_fiji_center_table(regions, path: str) -> None:
    """Save centroids (XM1, YM1, XM2...) = Fiji."""
    headers = []
    row_data = []
    for idx, r in enumerate(regions):
        roi_num = idx + 1
        headers.append(f"XM{roi_num}")
        headers.append(f"YM{roi_num}")
        cy, cx = r.centroid  # Python returns (Y, X), Fiji expects (X, Y)
        row_data.append(str(round(cx)))
        row_data.append(str(round(cy)))
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\t" + "\t".join(headers) + "\n")
        fh.write("1\t" + "\t".join(row_data) + "\n")


def save_fiji_intensity_table(intensity_matrix, path: str) -> None:
    """Save intensities (Mean1, Mean2...)"""
    num_rois = len(intensity_matrix)
    num_frames = len(intensity_matrix[0]) if num_rois > 0 else 0
    headers = [f"Mean{i+1}" for i in range(num_rois)]
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\t" + "\t".join(headers) + "\n")
        for t in range(num_frames):
            row_data = []
            for roi_idx in range(num_rois):
                val = intensity_matrix[roi_idx][t]
                row_data.append(str(int(val)))
            frame_num = t + 1
            fh.write(f"{frame_num}\t" + "\t".join(row_data) + "\n")


# EXPORT CONFIGURATION (DYNAMIC)
# Uses the folder selected in the button panel
OUTPUT_DIR = output_path
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Extract the video filename without extension to name the output text files (e.g. "calcium512")
stem_name = os.path.splitext(os.path.basename(movie_path))[0]

print(f"Starting file export to: {OUTPUT_DIR}")

# Assuming 'movie' is the original stack (Frames x H x W)
max_projection = movie.max(axis=0)

# Convert to 8-bit for visual compatibility with FIJI (.png)
mn, mx = max_projection.min(), max_projection.max()
if mx > mn:
    max_projection_8bit = (
        (max_projection - mn) / (mx - mn) * 255.0
    ).astype(np.uint8)
else:
    max_projection_8bit = max_projection.astype(np.uint8)

# (_max projection.png)
png_path = os.path.join(OUTPUT_DIR, f"{stem_name}_max projection.png")
iio.imwrite(png_path, max_projection_8bit)
print(f"  -> Maximum Projection saved at: {png_path}")


# (_center.txt)
caiman_regions = []

for mask_bin in mask_list:
    lbl = label(mask_bin)
    if np.any(lbl):
        caiman_regions.append(regionprops(lbl)[0])
    else:

        class DummyRegion:
            centroid = (0, 0)

        caiman_regions.append(DummyRegion())

centroid_path = os.path.join(OUTPUT_DIR, f"{stem_name}_center.txt")
# Corrected spelling error from the original script (save_fiji_center_table)
save_fiji_center_table(caiman_regions, centroid_path)
print(f"  -> ROI Centroids saved at: {centroid_path}")


# (_mean intensities.txt)
intensity_path = os.path.join(OUTPUT_DIR, f"{stem_name}_mean intensities.txt")
save_fiji_intensity_table(caiman_pure_means, intensity_path)
print(f"  -> Mean Intensities saved at: {intensity_path}")

print("\n[SUCCESS] All files exported in standard FIJI format!")

#### 2. (Exploratory) Save scripts for Connectivity Analysis based on frequency (Traces, using Masks from CaIman & Traces from CaIman)

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import scipy.io as sio

# 1. UPDATE INDICES WITH YOUR MANUAL SELECTION (from the previous step)
idx_accepted_final = [
    id_trace for id_trace, keep in selection_state.items() if keep
]

# Update the accepted components in the object to match the export
cnmf_refit.estimates.idx_components = np.array(idx_accepted_final)

# 2. FILTER TRACES (C) AND SPATIAL MATRIX (A) TO KEEP ONLY THE ACCEPTED COMPONENTS
C_accepted = cnmf_refit.estimates.C[cnmf_refit.estimates.idx_components, :]
A_accepted = cnmf_refit.estimates.A[:, cnmf_refit.estimates.idx_components]

# 3. EXTRACT THE BACKGROUND (f and b from the refit for consistency)
f = cnmf_refit.estimates.f
b = cnmf_refit.estimates.b

# 4. EXPORT EVERYTHING TO MAT-FILES (IN THE DYNAMIC OUTPUT FOLDER)
# Define full paths pointing to the output folder
path_C = os.path.join(output_path, "C_accepted.mat")
path_A = os.path.join(output_path, "A_accepted.mat")
path_f = os.path.join(output_path, "f.mat")
path_b = os.path.join(output_path, "b.mat")

sio.savemat(path_C, {"C_accepted": C_accepted})
# .toarray() applied here to convert the sparse matrix to dense for MATLAB
sio.savemat(path_A, {"A_accepted": A_accepted.toarray()})
sio.savemat(path_f, {"f": f})
sio.savemat(path_b, {"b": b})

print("==========================================================")
print(
    f"Export completed successfully! {len(idx_accepted_final)} components saved."
)
print(f"Files saved at: {output_path}")
print("==========================================================")

In [ ]:
#Overview of Trace to be used in Frequency Connectivity Analysis - Aarhus pipeline

num_GBM = 2

# 3. Criar o gráfico
plt.figure(figsize=(12, 6))

for i in range(num_GBM):
    # Plota o traço de cada GBM com um pequeno desfasamento vertical (offset) 
    # para que não fiquem todos sobrepostos uns em cima dos outros
    plt.plot(C_accepted[i, :] + (i * 500), label=f'GBM {i}')

plt.title("Visualização Rápida dos Traços C")
plt.xlabel("Frames")
plt.ylabel("Fluorescência (u.a. com offset)")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()